# Notebook 02 — News Landscape Analysis

**Project:** News Pulse  
**Data source:** GDELT 2.0 Global Knowledge Graph (GKG)  
**Warehouse:** `news_pulse` MySQL database  
**Tools:** Python, SQLAlchemy, pandas, matplotlib, seaborn

---

## What this notebook covers

1. Segment volume and distribution
2. Sentiment patterns by segment
3. Publication volume over time
4. Source diversity — how many outlets cover each segment
5. Publication velocity — how quickly stories spread across sources
6. Cross-segment entity tracking — people and organisations appearing across multiple segments

---

## Methodology notes

Two limitations apply to this analysis and are worth stating upfront.

**Segment skew in historical data.** The `SEGMENT_MAP` in `transform.py` was expanded mid-project to cover more GDELT theme tag prefixes (e.g. `SOC_GENERALCRIME`, `ELECTION`, `WB_621_HEALTH`). Articles ingested before that patch were routed using the narrower original map, which left roughly 41% of the 11.9M historical articles in the General bucket. The fix applies to all articles ingested going forward. Analyses that break down volume by segment should be read with this in mind — General is inflated relative to what it would be with the corrected map applied consistently.

**Sentiment scored on source name, not headline.** The GDELT GKG feed does not carry article headlines. Sentiment is scored by VADER on the publication source name (e.g. `Reuters`, `BBC News`). This produces a measure of how emotionally charged a source's name reads, not the article content. Sentiment comparisons across segments reflect source-level patterns, not story-level tone. This is noted wherever sentiment findings appear.

---

## 0. Setup

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
from pathlib import Path

# Locate project root and load credentials
def find_project_root() -> Path:
    current = Path().resolve()
    while current != current.parent:
        if (current / "environment.yml").exists():
            return current
        current = current.parent
    raise FileNotFoundError("Could not locate project root.")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME", "news_pulse")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True
)

# Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 5)

print("Setup complete.")
print(f"Project root: {PROJECT_ROOT}")

## 1. Segment Volume and Distribution

How many articles landed in each segment? This tells us where GDELT's coverage is concentrated and how much analytical weight each segment can carry.

Note: General is inflated due to the historical segment skew described in the methodology notes above.

In [ ]:
query = """
SELECT
    ds.segment_name,
    COUNT(*) AS article_count
FROM fact_articles fa
JOIN dim_segment ds ON fa.segment_id = ds.segment_id
GROUP BY ds.segment_name
ORDER BY article_count DESC
"""

df_segments = pd.read_sql(query, engine)
df_segments["pct"] = (df_segments["article_count"] / df_segments["article_count"].sum() * 100).round(1)

print(df_segments.to_string(index=False))

In [ ]:
fig, ax = plt.subplots()

bars = ax.barh(
    df_segments["segment_name"],
    df_segments["article_count"],
    color=sns.color_palette("muted", len(df_segments))
)

# Label each bar with count and percentage
for bar, (_, row) in zip(bars, df_segments.iterrows()):
    ax.text(
        bar.get_width() + df_segments["article_count"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{row['article_count']:,}  ({row['pct']}%)",
        va="center", fontsize=9
    )

ax.set_xlabel("Article count")
ax.set_title("Article volume by segment\n(General inflated — historical segment skew applies)", fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "01_segment_volume.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/01_segment_volume.png")

## 2. Sentiment Patterns by Segment

Average VADER compound sentiment score per segment.

**Reminder:** Sentiment is scored on source name, not article content. Treat this as a source-level signal, not a story-level one. The variation across segments reflects which types of outlets tend to cover each topic, not the emotional tone of the articles themselves.

In [ ]:
# Section 2 — Sentiment Analysis: Abandoned
#
# VADER sentiment was scored on publication source names (e.g. "yahoo.com",
# "iheart.com", "indiatimes.com") because GDELT GKG does not carry article
# headlines or body text. Domain strings carry no emotionally loaded words,
# so VADER returned 0.0 compound score and 100% neutral label across every
# segment. The data confirmed this: avg_sentiment = 0.0, std = 0.0 for all
# seven segments with no variation whatsoever.
#
# Sentiment analysis is not presented as a finding in this notebook.
#
# What would be needed to do this properly:
#   - Article headlines or body text, obtained by scraping each URL
#   - This is out of scope for the pipeline as designed — GDELT is used
#     as a metadata feed, not a full-text source
#
# If sentiment analysis is added in a future version, the scoring should
# run on scraped headline text and be stored as a separate field in
# fact_articles rather than derived from source_name.

print("Sentiment analysis abandoned — see comment above for reasoning.")
print("No chart produced for this section.")

## 3. Publication Volume Over Time

Monthly article volume, broken down by segment. Shows whether coverage is growing, flat, or declining, and whether any segments are seasonal.

In [ ]:
query = """
SELECT
    dd.year,
    dd.month,
    ds.segment_name,
    COUNT(*) AS article_count
FROM fact_articles fa
JOIN dim_date    dd ON fa.date_id    = dd.date_id
JOIN dim_segment ds ON fa.segment_id = ds.segment_id
GROUP BY dd.year, dd.month, ds.segment_name
ORDER BY dd.year, dd.month, ds.segment_name
"""

df_time = pd.read_sql(query, engine)
df_time["month_dt"] = pd.to_datetime(
    df_time["year"].astype(str) + "-" + df_time["month"].astype(str).str.zfill(2) + "-01"
)

print(f"Date range: {df_time['month_dt'].min().date()} to {df_time['month_dt'].max().date()}")
print(f"Months covered: {df_time['month_dt'].nunique()}")

In [ ]:
# Total volume across all segments
df_total = df_time.groupby("month_dt")["article_count"].sum().reset_index()

# Total volume — with backfill boundary annotation
fig, ax = plt.subplots()

ax.plot(
    df_total["month_dt"], df_total["article_count"],
    marker="o", markersize=4, linewidth=1.5
)

# Mark where backfill ends and live pipeline begins
# Backfill covered Feb–May 2025. Live pipeline started from that point.
backfill_end = pd.Timestamp("2025-05-01")
ax.axvline(backfill_end, color="#d9534f", linewidth=1.2, linestyle="--")
ax.text(
    backfill_end + pd.Timedelta(days=10), ax.get_ylim()[1] * 0.85,
    "← Backfill ends\n   Live pipeline →",
    color="#d9534f", fontsize=8.5
)

ax.set_xlabel("Month")
ax.set_ylabel("Article count")
ax.set_title(
    "Total article volume per month (all segments)\n"
    "Note: Feb–May 2025 is backfill (~130k articles/day). "
    "Post-May 2025 is live pipeline (~15k/day).\n"
    "The apparent decline is a data collection artefact, not a real trend."
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "03_monthly_volume_total.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Volume by segment over time — exclude General to keep the chart readable
df_by_seg = df_time[df_time["segment_name"] != "General"].copy()

# Per-segment volume — same annotation, General excluded
fig, ax = plt.subplots(figsize=(14, 6))

for seg, grp in df_by_seg.groupby("segment_name"):
    grp_sorted = grp.sort_values("month_dt")
    ax.plot(
        grp_sorted["month_dt"], grp_sorted["article_count"],
        marker="o", markersize=3, linewidth=1.2, label=seg
    )

ax.axvline(backfill_end, color="#d9534f", linewidth=1.2, linestyle="--")
ax.text(
    backfill_end + pd.Timedelta(days=10), ax.get_ylim()[1] * 0.85,
    "← Backfill ends\n   Live pipeline →",
    color="#d9534f", fontsize=8.5
)

ax.set_xlabel("Month")
ax.set_ylabel("Article count")
ax.set_title(
    "Monthly article volume by segment (General excluded)\n"
    "Volume drop after May 2025 reflects pipeline mode change, not reduced coverage."
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(fontsize=8, loc="upper right")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "04_monthly_volume_by_segment.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Source Diversity

How many distinct outlets cover each segment? A segment covered by 3 sources is far more concentrated — and potentially less reliable as a signal — than one covered by 300.

We also look at the top sources per segment to see which outlets dominate each area.

In [ ]:
query = """
SELECT
    ds.segment_name,
    COUNT(DISTINCT fa.source_id)                                   AS unique_sources,
    COUNT(*)                                                       AS total_articles,
    ROUND(COUNT(*) / COUNT(DISTINCT fa.source_id), 1)             AS avg_articles_per_source
FROM fact_articles fa
JOIN dim_segment ds ON fa.segment_id = ds.segment_id
GROUP BY ds.segment_name
ORDER BY unique_sources DESC
"""

df_diversity = pd.read_sql(query, engine)
print(df_diversity.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Unique sources per segment
axes[0].barh(
    df_diversity["segment_name"],
    df_diversity["unique_sources"],
    color=sns.color_palette("muted", len(df_diversity))
)
axes[0].set_xlabel("Unique sources")
axes[0].set_title("Source diversity by segment")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[0].invert_yaxis()

# Avg articles per source
axes[1].barh(
    df_diversity["segment_name"],
    df_diversity["avg_articles_per_source"],
    color=sns.color_palette("muted", len(df_diversity))
)
axes[1].set_xlabel("Average articles per source")
axes[1].set_title("Publication concentration by segment")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[1].invert_yaxis()

plt.suptitle("Source diversity and concentration", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "05_source_diversity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Top 5 sources per segment by article count
query = """
SELECT segment_name, source_name, article_count, rnk
FROM (
    SELECT
        ds.segment_name,
        src.source_name,
        COUNT(*) AS article_count,
        RANK() OVER (PARTITION BY ds.segment_name ORDER BY COUNT(*) DESC) AS rnk
    FROM fact_articles fa
    JOIN dim_segment ds  ON fa.segment_id = ds.segment_id
    JOIN dim_source  src ON fa.source_id  = src.source_id
    GROUP BY ds.segment_name, src.source_name
) ranked
WHERE rnk <= 5
ORDER BY segment_name, rnk
"""

df_top_sources = pd.read_sql(query, engine)
print(df_top_sources.to_string(index=False))

## 5. Publication Velocity

How quickly does a story spread across multiple sources within the same day? A high number of sources covering the same segment in a short window suggests a breaking news event. Low velocity suggests niche or slow-burn topics.

We measure this as the average number of distinct sources publishing per day per segment.

In [ ]:
query = """
SELECT
    ds.segment_name,
    dd.full_date,
    COUNT(DISTINCT fa.source_id) AS sources_per_day,
    COUNT(*)                     AS articles_per_day
FROM fact_articles fa
JOIN dim_date    dd ON fa.date_id    = dd.date_id
JOIN dim_segment ds ON fa.segment_id = ds.segment_id
GROUP BY ds.segment_name, dd.full_date
"""

df_velocity = pd.read_sql(query, engine)

# Aggregate to avg per segment
df_vel_summary = (
    df_velocity
    .groupby("segment_name")
    .agg(
        avg_sources_per_day=("sources_per_day", "mean"),
        avg_articles_per_day=("articles_per_day", "mean"),
        max_sources_single_day=("sources_per_day", "max")
    )
    .round(1)
    .reset_index()
    .sort_values("avg_sources_per_day", ascending=False)
)

print(df_vel_summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots()

bars = ax.barh(
    df_vel_summary["segment_name"],
    df_vel_summary["avg_sources_per_day"],
    color=sns.color_palette("muted", len(df_vel_summary))
)

for bar, (_, row) in zip(bars, df_vel_summary.iterrows()):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"max: {int(row['max_sources_single_day'])}",
        va="center", fontsize=8, color="#555555"
    )

ax.set_xlabel("Average unique sources per day")
ax.set_title("Publication velocity by segment\n(max = peak sources in one day)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "06_publication_velocity.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Cross-Segment Entity Tracking

Which people and organisations appear across the most segments? An entity appearing in Politics, Business, and Science simultaneously is a different kind of signal than one that only appears in a single segment.

This is the cross-segment presence score: count of distinct segments an entity appears in.

In [ ]:
query = """
SELECT
    de.entity_name,
    de.entity_type,
    COUNT(DISTINCT fa.segment_id)  AS segment_count,
    COUNT(*)                       AS total_mentions
FROM fact_entity_mentions fem
JOIN dim_entity     de  ON fem.entity_id  = de.entity_id
JOIN fact_articles  fa  ON fem.article_id = fa.article_id
GROUP BY de.entity_name, de.entity_type
HAVING segment_count >= 3
ORDER BY segment_count DESC, total_mentions DESC
LIMIT 50
"""

df_cross = pd.read_sql(query, engine)
print(f"Entities appearing in 3+ segments: {len(df_cross)}")
print(df_cross.head(20).to_string(index=False))

In [ ]:
# Split by entity type and plot top 15 per type
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, etype in zip(axes, ["PERSON", "ORG"]):
    df_sub = (
        df_cross[df_cross["entity_type"] == etype]
        .head(15)
        .sort_values("total_mentions")
    )
    bars = ax.barh(
        df_sub["entity_name"],
        df_sub["total_mentions"],
        color=sns.color_palette("muted", len(df_sub))
    )
    for bar, (_, row) in zip(bars, df_sub.iterrows()):
        ax.text(
            bar.get_width() + df_sub["total_mentions"].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{int(row['segment_count'])} segs",
            va="center", fontsize=8
        )
    ax.set_xlabel("Total mentions")
    ax.set_title(f"Top 15 cross-segment {etype}s\n(label = segment count)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.suptitle("Entities appearing across 3+ segments", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "07_cross_segment_entities.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Segment Breakdown: Which Segments Share the Most Entities?

A pairwise look at entity overlap between segments. High overlap between two segments means they share the same cast of people and organisations — useful context for the recommendation engine.

In [ ]:
query = """
CREATE TEMPORARY TABLE IF NOT EXISTS tmp_entity_segment AS
SELECT DISTINCT
    fem.entity_id,
    fa.segment_id
FROM fact_entity_mentions fem
JOIN fact_articles fa ON fem.article_id = fa.article_id
"""

with engine.begin() as conn:
    conn.execute(text(query))
print("Temporary table built.")

In [ ]:
df_es = pd.read_sql("SELECT entity_id, segment_id FROM tmp_entity_segment", engine)

# Join segment names in Python
seg_names = pd.read_sql("SELECT segment_id, segment_name FROM dim_segment", engine)
df_es = df_es.merge(seg_names, on="segment_id")

print(f"Entity-segment pairs: {len(df_es):,}")
print(f"Unique entities: {df_es['entity_id'].nunique():,}")
print(f"Unique segments: {df_es['segment_name'].nunique()}")

In [ ]:
from itertools import combinations

# Group segments per entity
entity_to_segs = df_es.groupby("entity_id")["segment_name"].apply(set)

# Count shared entities per segment pair
from collections import defaultdict
pair_counts = defaultdict(int)

for segs in entity_to_segs:
    if len(segs) >= 2:
        for a, b in combinations(sorted(segs), 2):
            pair_counts[(a, b)] += 1

# Build matrix
segments = sorted(df_es["segment_name"].unique())
matrix = pd.DataFrame(0, index=segments, columns=segments)

for (a, b), count in pair_counts.items():
    matrix.loc[a, b] = count
    matrix.loc[b, a] = count

print("Overlap matrix:")
print(matrix.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    matrix, annot=True, fmt=",d", cmap="Blues",
    linewidths=0.5, ax=ax
)
ax.set_title("Entity overlap between segments\n(shared unique entities per pair)")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "08_entity_overlap_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Summary Findings

Replace the placeholder text below after running each section. Write what the data actually showed, not what you expected.

## Summary Findings

**Segment distribution**
General accounts for 41% of all articles (4.88M), inflated by the historical segment skew prior to the SEGMENT_MAP patch. Among named segments, Politics and Government is the largest at 25.5% (3.04M), followed by Entertainment and Culture (11.2%), Science and Health (10.6%), and Business and Markets (9.9%). Crime and Justice and Technology sit well under 1% each — low article counts that are consistent with how infrequently GDELT assigns those theme tags, not a pipeline failure.

**Sentiment**
Sentiment analysis was abandoned. VADER scored all source names as exactly neutral (0.0 compound, 100% neutral label, zero variance across all segments) because GDELT GKG does not carry headline or body text — only publication domain names. Scoring sentiment on "yahoo.com" or "iheart.com" produces no useful signal. Any future sentiment layer would require scraping article headlines directly.

**Volume over time**
The apparent decline in article volume after May 2025 is a data collection artefact, not a real trend. The backfill processed roughly 130,000 articles per day across Feb–May 2025. The live pipeline processes roughly 15,000 per day from May 2025 onward. Within the backfill period, Politics and Government shows a clear spike in March–April 2025 that the other segments do not, likely corresponding to specific political events in that window.

**Source diversity**
The top five segments each draw from 10,500–14,500 unique sources, which is healthy coverage breadth. Technology and Crime sit at 6,600 and 5,800 respectively, consistent with their lower volumes. Concentration tells a different story: Politics averages 279 articles per source, nearly double Entertainment (123) and Business (112), meaning political coverage comes from outlets publishing at much higher volume. The top sources across most segments are aggregator platforms — iHeart, Yahoo, India Times, tvguide.co.uk — rather than specialist beat reporters. iHeart alone contributes 413,791 articles to General and 87,989 to Entertainment.

**Publication velocity**
General and Politics lead on daily source spread (5,663 and 3,963 average unique sources per day respectively), with peak single-day counts of 6,753 and 4,935. Crime and Technology average under 720 sources per day and peaked below 1,200, reflecting how concentrated coverage in those segments is. High velocity in Politics relative to its source pool suggests a smaller number of outlets publishing political content at very high frequency.

**Cross-segment entities**
Donald Trump (1.13M mentions) and United States as an organisation (1.05M mentions) are the two most-mentioned entities in the dataset, each appearing across all seven segments. Two city names — Los Angeles and Las Vegas — appear incorrectly classified as PERSON entities, a spaCy NER error from GDELT's persons field. These are noise in the entity data, not meaningful signal.

The entity overlap heatmap shows the strongest cross-segment connection between General and Politics (671,419 shared entities), which partly reflects the known segment skew routing political content into General. General–Entertainment is the second strongest pair (514,806). Technology–Crime is the most distinct pairing in the dataset at 12,559 shared entities, meaning those two segments cover genuinely different subject matter with almost no shared cast of people or organisations. This has direct implications for the recommendation engine: cross-segment recommendations between Technology and Crime are unlikely to be relevant, while General–Politics and General–Entertainment recommendations may be highly relevant even when the segment labels differ.